# Model 2: Multi-resolution CNN Fingerprint Predictor

## 1. Environment Setup and Data Loading

This notebook implements a multi-resolution 1D CNN for
predicting molecular fingerprints from MS/MS spectra.

The model uses parallel convolutional branches with
different kernel sizes to capture spectral patterns
at multiple scales.

For a fair comparison with the standard 1D CNN, we use:

- The same training and validation datasets
- The same 2200-dimensional spectrum representation
- The same 1024-bit Morgan fingerprint targets
- The same weighted BCE loss
- The same learning rate and training protocol

We first verify that the saved prototype datasets
can be loaded independently.

In [ ]:
!pip -q install "rdkit==2026.3.3" pyarrow

In [ ]:
import json
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from rdkit import rdBase

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

def find_repo_root(start=None):
    current = Path(start or Path.cwd()).expanduser().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current

# Set these environment variables to override the local defaults.
REPO_ROOT = Path(
    os.environ.get("CASMI_REPO_ROOT", find_repo_root())
).expanduser().resolve()
SAVE_DIR = Path(
    os.environ.get(
        "CASMI_MEMBER2_DIR",
        REPO_ROOT / "outputs" / "member2"
    )
).expanduser().resolve()
SAVE_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = SAVE_DIR / "prototype_data"
print("Member 2 working directory:", SAVE_DIR)

# Load datasets
df_train = pd.read_parquet(
    DATA_DIR / "train_prototype.parquet"
)

df_val = pd.read_parquet(
    DATA_DIR / "val_prototype.parquet"
)

# Load structure split
with open(SAVE_DIR / "structure_split_scoring_safe.json") as f:
    split_info = json.load(f)

train_ids = set(split_info["train_structures"])
val_ids = set(split_info["val_structures"])

# Validate data
assert rdBase.rdkitVersion == "2026.03.3"
assert len(df_train) == 17456
assert len(df_val) == 2000

assert set(df_train["inchikey14"]).issubset(train_ids)
assert set(df_val["inchikey14"]).issubset(val_ids)

assert set(df_train["inchikey14"]).isdisjoint(
    set(df_val["inchikey14"])
)

print("Device:", device)
print("PyTorch version:", torch.__version__)
print("RDKit version:", rdBase.rdkitVersion)

print("\nTraining spectra:", len(df_train))
print("Validation spectra:", len(df_val))

print("\nMulti-resolution CNN data loaded successfully!")

## 2. PyTorch Dataset and DataLoader

We use the same data preprocessing as the 1D CNN baseline
to ensure a fair comparison.

### Input Representation
- 2200-dimensional binned MS/MS spectrum
- m/z range: [0, 2200)
- Bin width: 1 Da
- Maximum intensity aggregation
- Square-root intensity transformation

### Target Representation
- 1024-bit Morgan fingerprint
- Radius: 2
- Chirality: disabled

The dataset applies spectrum binning on demand and
caches molecular fingerprints to reduce repeated calculations.

We use a batch size of 32 for both training and validation.

In [ ]:
from functools import lru_cache

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

# Configuration
MZ_MAX = 2200
BIN_WIDTH = 1.0

FP_SIZE = 1024
FP_RADIUS = 2

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=FP_RADIUS,
    fpSize=FP_SIZE,
    includeChirality=False
)


def spectrum_to_bins(mzs, intensities):
    """Convert MS/MS peaks to a fixed-length spectrum vector."""

    mzs = np.asarray(mzs, dtype=np.float64)
    intensities = np.asarray(intensities, dtype=np.float32)

    if len(mzs) != len(intensities):
        raise ValueError("Mismatched spectrum array lengths")

    vector = np.zeros(MZ_MAX, dtype=np.float32)

    valid = (
        np.isfinite(mzs)
        & np.isfinite(intensities)
        & (mzs >= 0)
        & (mzs < MZ_MAX)
        & (intensities >= 0)
    )

    indices = np.floor(
        mzs[valid] / BIN_WIDTH
    ).astype(np.int64)

    values = np.sqrt(intensities[valid])

    np.maximum.at(vector, indices, values)

    return vector


@lru_cache(maxsize=30000)
def smiles_to_fingerprint(smiles):
    """Convert SMILES into a 1024-bit Morgan fingerprint."""

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fp = morgan_generator.GetFingerprint(mol)

    arr = np.zeros(FP_SIZE, dtype=np.uint8)

    DataStructs.ConvertToNumpyArray(fp, arr)

    return arr


print("Preprocessing functions ready!")

In [ ]:
from torch.utils.data import DataLoader, Dataset


class SpectrumFingerprintDataset(Dataset):

    def __init__(self, dataframe):
        self.records = dataframe.to_dict("records")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        row = self.records[idx]

        # Spectrum input
        spectrum = spectrum_to_bins(
            row["ms2_mzs"],
            row["ms2_normalized_intensities"]
        )

        # Fingerprint target
        fingerprint = smiles_to_fingerprint(
            row["normalized_smiles"]
        )

        x = torch.from_numpy(spectrum).unsqueeze(0)

        y = torch.from_numpy(
            fingerprint.astype(np.float32)
        )

        return x, y


train_dataset = SpectrumFingerprintDataset(df_train)
val_dataset = SpectrumFingerprintDataset(df_val)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

print("\nDataLoaders created successfully!")

In [ ]:
X_batch, y_batch = next(iter(train_loader))

print("Input shape:", X_batch.shape)
print("Target shape:", y_batch.shape)

print("\nInput dtype:", X_batch.dtype)
print("Target dtype:", y_batch.dtype)

print("\nInput range:",
      X_batch.min().item(),
      X_batch.max().item())

print("Target unique values:", torch.unique(y_batch))

assert X_batch.shape == (32, 1, 2200)
assert y_batch.shape == (32, 1024)

assert torch.isfinite(X_batch).all()
assert torch.isfinite(y_batch).all()

assert X_batch.min() >= 0
assert X_batch.max() <= 1

assert torch.all((y_batch == 0) | (y_batch == 1))

print("\nAll DataLoader tests passed!")

## 3. Multi-resolution CNN Architecture

We build a multi-resolution CNN to predict molecular
fingerprints from MS/MS spectra.

Unlike the standard 1D CNN, this model uses three parallel
convolutional branches with different kernel sizes:

- Kernel size 3: captures local spectral patterns
- Kernel size 7: captures medium-range patterns
- Kernel size 15: captures wider-range patterns

Each branch produces 16 feature maps. The outputs are
concatenated and passed through additional convolutional
layers for feature fusion.

### Architecture

1. Input: (Batch, 1, 2200)
2. Three parallel Conv1d branches (3, 7, 15)
3. Concatenate feature maps (48 channels)
4. MaxPooling
5. Conv1d (48 → 64 channels, kernel size 5)
6. BatchNorm + ReLU + MaxPooling
7. Conv1d (64 → 128 channels, kernel size 3)
8. BatchNorm + ReLU
9. Adaptive Average Pooling
10. Fully Connected Layers
11. Output: 1024 fingerprint logits

We keep the classifier similar to the baseline CNN
to make the parameter counts approximately comparable.

In [ ]:
import torch

torch.manual_seed(42)


class MultiResolutionCNNFingerprint(nn.Module):

    def __init__(self, fp_size=1024):
        super().__init__()

        # Parallel convolutional branches
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(
                    in_channels=1,
                    out_channels=16,
                    kernel_size=k,
                    padding=k // 2
                ),
                nn.BatchNorm1d(16),
                nn.ReLU()
            )
            for k in [3, 7, 15]
        ])

        # Feature fusion
        self.features = nn.Sequential(
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(
                in_channels=48,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            nn.Conv1d(
                in_channels=64,
                out_channels=128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(16)
        )

        # Fingerprint prediction head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, fp_size)
        )

    def forward(self, x):

        # Parallel feature extraction
        branch_outputs = [
            branch(x) for branch in self.branches
        ]

        # Concatenate along channel dimension
        x = torch.cat(branch_outputs, dim=1)

        # Feature fusion
        x = self.features(x)

        # Predict fingerprint logits
        x = self.classifier(x)

        return x


# Initialize model
model = MultiResolutionCNNFingerprint(
    fp_size=FP_SIZE
).to(device)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Device:", device)
print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")

print("\nModel architecture:")
print(model)

In [ ]:
# Test forward pass

model.eval()

X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

with torch.no_grad():
    logits = model(X_batch)
    probabilities = torch.sigmoid(logits)

print("Input shape:", X_batch.shape)
print("Target shape:", y_batch.shape)

print("Output logits shape:", logits.shape)

print("\nProbability range:")
print(
    probabilities.min().item(),
    probabilities.max().item()
)

print("\nOutput contains NaN:",
      torch.isnan(logits).any().item())

print("Output contains Inf:",
      torch.isinf(logits).any().item())

# Validate outputs
assert logits.shape == y_batch.shape
assert logits.shape == (32, 1024)

assert torch.isfinite(logits).all()

assert probabilities.min() >= 0
assert probabilities.max() <= 1

print("\nMulti-resolution CNN forward pass passed!")

## 4. Loss Function and Optimizer

We use the same loss function and optimizer as the
standard 1D CNN baseline for a fair comparison.

### Training Configuration
- Loss: Weighted BCEWithLogitsLoss
- Optimizer: AdamW
- Learning rate: 0.0003
- Weight decay: 0.0001
- Batch size: 32

Since Morgan fingerprints are sparse, positive class
weights are calculated from the training data.

The weights are capped at 20 to avoid excessively
large contributions from rare fingerprint bits.

We first test the backward pass before full training.

In [ ]:
from tqdm.auto import tqdm

# Count positive fingerprint bits
bit_counts = np.zeros(FP_SIZE, dtype=np.int64)

for smiles in tqdm(
    df_train["normalized_smiles"],
    desc="Calculating fingerprint frequencies"
):
    fp = smiles_to_fingerprint(smiles)
    bit_counts += fp.astype(np.int64)

n_train = len(df_train)

print("Training spectra:", n_train)
print("Average active bits:", bit_counts.sum() / n_train)
print("Bits never activated:", np.sum(bit_counts == 0))

# Same weighting policy as Model 1
pos_weights = np.ones(FP_SIZE, dtype=np.float32)

seen = bit_counts > 0

pos_weights[seen] = np.clip(
    (n_train - bit_counts[seen]) / bit_counts[seen],
    1.0,
    20.0
)

pos_weight_tensor = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device
)

print(
    "\nPositive weight range:",
    pos_weight_tensor.min().item(),
    pos_weight_tensor.max().item()
)

print("\nClass weights calculated successfully!")

In [ ]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor
)

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

print("Loss:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])

print("\nTraining configuration ready!")

In [ ]:
model.train()

X_batch, y_batch = next(iter(train_loader))

X_batch = X_batch.to(device)
y_batch = y_batch.to(device)

# Record weights before update
weights_before = (
    model.classifier[-1].weight.detach().clone()
)

# Forward pass
optimizer.zero_grad(set_to_none=True)

logits = model(X_batch)

# Compute weighted BCE loss
loss = criterion(logits, y_batch)

print("Initial training loss:", loss.item())

# Backward pass
loss.backward()

gradient_norm = (
    model.classifier[-1].weight.grad.norm().item()
)

print("Gradient norm:", gradient_norm)

# Update parameters
optimizer.step()

weight_change = (
    model.classifier[-1].weight.detach()
    - weights_before
).abs().mean().item()

print("Average weight change:", weight_change)

# Checks
assert torch.isfinite(loss).item()
assert gradient_norm > 0
assert weight_change > 0

print("\nForward, backward, and optimizer step passed!")

## 5. Training and Evaluation

We use the same training and evaluation procedure as
the standard 1D CNN baseline.

We monitor:

- Weighted BCE loss
- Micro precision, recall, and F1
- Mean fingerprint Tanimoto similarity
- Predicted positive-bit rate
- Actual positive-bit rate

Both models use the same validation dataset and
training hyperparameters for a fair comparison.

The model is evaluated on the full validation set
after each training epoch.

In [ ]:
from tqdm.auto import tqdm


def run_epoch(model, loader, criterion, optimizer=None,
              max_batches=None):

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    tp = 0
    fp = 0
    fn = 0

    predicted_positive = 0
    actual_positive = 0
    total_bits = 0
    tanimoto_sum = 0.0

    for batch_idx, (x, y) in enumerate(loader):

        if max_batches is not None and batch_idx >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            logits = model(x)
            loss = criterion(logits, y)

            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = x.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        with torch.no_grad():
            probs = torch.sigmoid(logits)

            preds = probs >= 0.5
            targets = y >= 0.5

            tp += (preds & targets).sum().item()
            fp += (preds & ~targets).sum().item()
            fn += (~preds & targets).sum().item()

            predicted_positive += preds.sum().item()
            actual_positive += targets.sum().item()
            total_bits += targets.numel()

            intersection = (
                preds & targets
            ).sum(dim=1).float()

            union = (
                preds | targets
            ).sum(dim=1).float()

            tanimoto = torch.where(
                union > 0,
                intersection / union.clamp(min=1),
                torch.ones_like(union)
            )

            tanimoto_sum += tanimoto.sum().item()

    if total_samples == 0:
        raise ValueError("No batches were processed")

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    return {
        "loss": total_loss / total_samples,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tanimoto": tanimoto_sum / total_samples,
        "pred_positive_rate": predicted_positive / total_bits,
        "actual_positive_rate": actual_positive / total_bits
    }


print("Training and evaluation functions ready!")

## 6. Full Prototype Training

We train the Multi-resolution CNN on the same
17,456-spectrum prototype dataset used by Model 1.

The training configuration is identical to the
standard CNN baseline.

We will:

- Reinitialize model parameters with seed 42
- Train using all 546 training batches per epoch
- Evaluate on all 63 validation batches
- Save the checkpoint with the lowest validation loss
- Apply early stopping with patience 3
- Save training metrics for model comparison

This experiment evaluates the effect of the
multi-resolution architecture while keeping
the remaining training settings unchanged.

In [ ]:
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Reinitialize model after backward pass test
model = MultiResolutionCNNFingerprint(
    fp_size=FP_SIZE
).to(device)

# Same loss configuration as Model 1
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight_tensor.to(device)
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

# Training configuration
EPOCHS = 8
PATIENCE = 3

best_val_loss = float("inf")
epochs_without_improvement = 0

history_full = []

# Model 2 checkpoint path
CKPT_DIR = SAVE_DIR / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_model_path = (
    CKPT_DIR / "multires_cnn_best_val_loss.pt"
)

print("Starting Multi-resolution CNN training...")
print("Device:", device)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

for epoch in range(1, EPOCHS + 1):

    start_time = time.perf_counter()

    # Training
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer
    )

    # Validation
    val_metrics = run_epoch(
        model,
        val_loader,
        criterion,
        optimizer=None
    )

    elapsed = time.perf_counter() - start_time

    history_full.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_f1": train_metrics["f1"],
        "val_loss": val_metrics["loss"],
        "val_f1": val_metrics["f1"],
        "val_tanimoto": val_metrics["tanimoto"],
        "val_pred_positive_rate": val_metrics["pred_positive_rate"],
        "val_actual_positive_rate": val_metrics["actual_positive_rate"],
        "seconds": elapsed
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Tanimoto: {val_metrics['tanimoto']:.4f} | "
        f"Pred Positive: {val_metrics['pred_positive_rate']:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    # Save best checkpoint
    if val_metrics["loss"] < best_val_loss - 1e-4:

        best_val_loss = val_metrics["loss"]
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_metrics["loss"],
            "val_f1": val_metrics["f1"],
            "val_tanimoto": val_metrics["tanimoto"],
            "fp_size": FP_SIZE,
            "mz_max": MZ_MAX,
            "architecture": "MultiResolutionCNNFingerprint"
        }, best_model_path)

        print("  -> Best model saved!")

    else:
        epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            print("\nEarly stopping triggered.")
            break

# Save training history
history_full_df = pd.DataFrame(history_full)

history_path = (
    SAVE_DIR / "multires_cnn_training_history.csv"
)

history_full_df.to_csv(
    history_path,
    index=False
)

print("\nTraining Summary:")
display(history_full_df)

print("\nBest validation loss:", best_val_loss)
print("Best model:", best_model_path)
print("Training history:", history_path)

print("\nMulti-resolution CNN training completed!")

## 7. Best Checkpoint Evaluation and Threshold Analysis

The Multi-resolution CNN completed eight training epochs.

The best checkpoint was obtained at epoch 6, with:
- Validation loss: 0.8478
- Micro F1 at threshold 0.5: 0.2653
- Mean Tanimoto at threshold 0.5: 0.1550

We load the best checkpoint and evaluate different
fingerprint probability thresholds.

Metrics:
- Micro precision
- Micro recall
- Micro F1
- Mean Tanimoto similarity
- Predicted positive-bit rate

We use the same threshold values and evaluation
procedure as the standard 1D CNN for comparison.

In [ ]:
# Load best Multi-resolution CNN checkpoint

checkpoint = torch.load(
    best_model_path,
    map_location=device,
    weights_only=True
)

best_model = MultiResolutionCNNFingerprint(
    fp_size=FP_SIZE
).to(device)

best_model.load_state_dict(
    checkpoint["model_state_dict"]
)

best_model.eval()

print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Loss:", checkpoint["val_loss"])
print("Validation F1:", checkpoint["val_f1"])
print("Validation Tanimoto:", checkpoint["val_tanimoto"])

assert checkpoint["epoch"] == 6

print("\nBest checkpoint loaded successfully!")

In [ ]:
from tqdm.auto import tqdm

best_model.eval()

all_probs = []
all_targets = []

with torch.inference_mode():

    for x, y in tqdm(
        val_loader,
        desc="Evaluating Multi-resolution CNN"
    ):

        x = x.to(device)

        logits = best_model(x)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu())
        all_targets.append(y.cpu())

val_probs = torch.cat(all_probs, dim=0)
val_targets = torch.cat(all_targets, dim=0)

print("Predictions shape:", val_probs.shape)
print("Targets shape:", val_targets.shape)

assert val_probs.shape == (2000, 1024)
assert val_targets.shape == (2000, 1024)
assert torch.isfinite(val_probs).all()

print("\nValidation predictions ready!")

In [ ]:
threshold_results = []

targets = val_targets >= 0.5

thresholds = np.round(
    np.arange(0.20, 0.81, 0.05),
    2
)

for threshold in thresholds:

    preds = val_probs >= threshold

    tp = (preds & targets).sum().item()
    fp = (preds & ~targets).sum().item()
    fn = (~preds & targets).sum().item()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    f1 = (
        2 * precision * recall
        / (precision + recall + 1e-8)
    )

    # Per-spectrum Tanimoto
    intersection = (
        preds & targets
    ).sum(dim=1).float()

    union = (
        preds | targets
    ).sum(dim=1).float()

    tanimoto = torch.where(
        union > 0,
        intersection / union.clamp(min=1),
        torch.ones_like(union)
    ).mean().item()

    pred_positive_rate = preds.float().mean().item()

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tanimoto": tanimoto,
        "pred_positive_rate": pred_positive_rate
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df.round(4))

best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

best_tanimoto_row = threshold_df.loc[
    threshold_df["tanimoto"].idxmax()
]

print("\nBest threshold by F1:")
print(best_f1_row.round(4))

print("\nBest threshold by Tanimoto:")
print(best_tanimoto_row.round(4))

# Sanity check: threshold 0.5 matches saved checkpoint
f1_at_05 = threshold_df.loc[
    np.isclose(threshold_df["threshold"], 0.5),
    "f1"
].iloc[0]

assert np.isclose(
    f1_at_05,
    checkpoint["val_f1"],
    atol=1e-5
)

print("\nThreshold analysis completed!")

## 8. Save Multi-resolution CNN Evaluation Results

The threshold analysis is complete.

The Multi-resolution CNN achieved:
- Best validation loss: 0.8478 (epoch 6)
- Best Micro F1: 0.3099 (threshold 0.60)
- Best mean Tanimoto: 0.1834 (threshold 0.60)

We save the threshold results, validation predictions,
and evaluation summary.

The continuous fingerprint probabilities will be
used for candidate ranking and model comparison.

All results are based on the prototype validation set.

In [ ]:
import json
from pathlib import Path

import numpy as np

EVAL_DIR = SAVE_DIR / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save threshold analysis
threshold_path = (
    EVAL_DIR / "multires_cnn_threshold_results.csv"
)

threshold_df.to_csv(threshold_path, index=False)


# 2. Save continuous fingerprint predictions
predictions_path = (
    EVAL_DIR / "multires_cnn_val_predictions.npz"
)

assert len(val_probs) == len(df_val)
assert len(val_targets) == len(df_val)

np.savez_compressed(
    predictions_path,
    probabilities=val_probs.cpu().numpy().astype(np.float32),
    targets=val_targets.cpu().numpy().astype(np.uint8),
    inchikey14=df_val["inchikey14"].to_numpy(dtype=str),
    smiles=df_val["normalized_smiles"].to_numpy(dtype=str)
)


# 3. Save evaluation summary
evaluation_summary = {
    "model": "MultiResolutionCNNFingerprint",
    "checkpoint_epoch": int(checkpoint["epoch"]),
    "training_samples": len(df_train),
    "validation_samples": len(df_val),
    "fingerprint_size": FP_SIZE,
    "best_val_loss": float(checkpoint["val_loss"]),
    "f1_at_0.5": float(checkpoint["val_f1"]),
    "tanimoto_at_0.5": float(checkpoint["val_tanimoto"]),
    "best_f1_threshold": float(best_f1_row["threshold"]),
    "best_f1": float(best_f1_row["f1"]),
    "best_tanimoto_threshold": float(
        best_tanimoto_row["threshold"]
    ),
    "best_tanimoto": float(best_tanimoto_row["tanimoto"]),
    "ranking_input": "continuous fingerprint probabilities"
}

summary_path = (
    EVAL_DIR / "multires_cnn_evaluation_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(evaluation_summary, f, indent=2)


# 4. Verify saved files
assert threshold_path.exists()
assert predictions_path.exists()
assert summary_path.exists()

print("Threshold results:", threshold_path)
print("Validation predictions:", predictions_path)
print("Evaluation summary:", summary_path)

print("\nMulti-resolution CNN evaluation saved successfully!")

In [ ]:
# Verify that Model 1 and Model 2 predictions are aligned

model1_path = (
    EVAL_DIR / "cnn1d_val_predictions.npz"
)

model2_path = (
    EVAL_DIR / "multires_cnn_val_predictions.npz"
)

with np.load(model1_path, allow_pickle=False) as m1:
    model1_probs = m1["probabilities"]
    model1_targets = m1["targets"]
    model1_keys = m1["inchikey14"]
    model1_smiles = m1["smiles"]

with np.load(model2_path, allow_pickle=False) as m2:
    model2_probs = m2["probabilities"]
    model2_targets = m2["targets"]
    model2_keys = m2["inchikey14"]
    model2_smiles = m2["smiles"]

print("Model 1 predictions:", model1_probs.shape)
print("Model 2 predictions:", model2_probs.shape)

print(
    "Same structure order:",
    np.array_equal(model1_keys, model2_keys)
)

print(
    "Same SMILES order:",
    np.array_equal(model1_smiles, model2_smiles)
)

print(
    "Same fingerprint targets:",
    np.array_equal(model1_targets, model2_targets)
)

assert model1_probs.shape == model2_probs.shape
assert np.array_equal(model1_keys, model2_keys)
assert np.array_equal(model1_smiles, model2_smiles)
assert np.array_equal(model1_targets, model2_targets)

print("\nModel comparison alignment passed!")